# Data Cleaning Notebook

In this notebook, we load all the .json data files from the [Open Trivia Database](https://opentdb.com/api_config.php) and combine them into a single .csv file. We were only able to query a maximum of 50 questions/answers from the website, so we have many .json files containing data for only 50 questions, so we will combine these questions into a single, much larger table. We also clean up the data slightly so that we have the desired attribute columns, and we also add question ids so we can access the questions more easily in the future, and remove duplicates from the dataset (in case duplicate questions were produced during the process when downloading the randomized 50 questions per .json).

## Load the data

Load all the .json files that were downloaded from the [Open Trivia Database](https://opentdb.com/api_config.php). Each .json file contains information for 50 questions, and there may be duplicates between each file.

In [1]:
import os
import glob
import string
import random
import pandas as pd
import numpy as np

# Get all files ending in json in the data folder
path = os.getcwd()
extension = 'json'
os.chdir(f'{path}/data_split/')
result = glob.glob('*.{}'.format(extension))
print(result)

['data62.json', 'data65.json', 'data55.json', 'data73.json', 'data45.json', 'data33.json', 'data36.json', 'data20.json', 'data22.json', 'data6.json', 'data49.json', 'data66.json', 'data40.json', 'data60.json', 'data23.json', 'data30.json', 'data46.json', 'data44.json', 'data19.json', 'data71.json', 'data56.json', 'data16.json', 'data28.json', 'data24.json', 'data1.json', 'data64.json', 'data63.json', 'data43.json', 'data11.json', 'data54.json', 'data27.json', 'data70.json', 'data13.json', 'data9.json', 'data25.json', 'data32.json', 'data17.json', 'data5.json', 'data68.json', 'data18.json', 'data26.json', 'data47.json', 'data72.json', 'data8.json', 'data4.json', 'data48.json', 'data74.json', 'data52.json', 'data29.json', 'data7.json', 'data39.json', 'data38.json', 'data58.json', 'data50.json', 'data10.json', 'data42.json', 'data57.json', 'data75.json', 'data51.json', 'data21.json', 'data31.json', 'data2.json', 'data41.json', 'data15.json', 'data12.json', 'data34.json', 'data37.json', 'd

## Clean up the original data

In this section, we do data clean up on the .json files. We first read in all the .json files and combine them into a single pandas dataframe. The original data file contains all the incorrect answers in a list as a single attribute, and the correct answer as a separate attribute. However, since we want to make our trivia game multiple choice, this is not ideal. Therefore, we split up the list of incorrect answers into separate attributes, and combined it with the correct answer. We then shuffled the order of the answers and made them into separate attributes (so we have attributes A, B, C, and D one of which contains the correct answer and the remaining three are incorrect). We also keep track of the location of the correct answer so we can later score the player. Finally, we also assign each question a unique 7 character id so we can access the questions more easily in the future.

In [2]:
# Helper function to get unique ids for the questions
def unique_id(size):
    chars = list(set(string.ascii_uppercase + string.digits).difference('LIO01'))
    return ''.join(random.choices(chars, k=size))

In [3]:
# Look at the original attributes of a single file
# The current attributes are not ideal, so we will need to do some clean up
pd.DataFrame(pd.read_json(result[0])['results'].values.tolist()).head()

,category,type,difficulty,question,correct_answer,incorrect_answers
0,Entertainment: Video Games,multiple,medium,"What name did &quot;Mario&quot;, from &quot;Su...",Ossan,"[Jumpman, Mr. Video, Mario]"
1,Entertainment: Japanese Anime & Manga,multiple,medium,Who wrote and directed the animated movie &quo...,Hayao Miyazaki,"[Isao Takahata, Mamoru Hosoda, Hidetaka Miyazaki]"
2,Entertainment: Video Games,multiple,medium,What&#039;s the famous line Vaas says in &quot...,Did I ever tell you the definition of Insanity?,"[Have I failed to entertain you?, You&#039;re ..."
3,Geography,multiple,easy,What state is the largest state of the United ...,Alaska,"[California, Texas, Washington]"
4,Entertainment: Video Games,boolean,hard,Unturned originally started as a Roblox game.,True,[False]


In [4]:
# Read all files in the directory and put it into a single dataframe
questions = pd.DataFrame(columns=['category', 'type', 'difficulty', 'question', 'correct_answer', 'A', 'B', 'C', 'D'])
for file in result:
    questions = questions.append(pd.DataFrame(pd.read_json(file)['results'].values.tolist()))

# Keep a counter
count = 0

# Create lists to store the incorrect answers and the correct answers 
incorrect = []
incorrect_answers = questions['incorrect_answers'].copy()
correct_answer_number=[]
answer_choices = ['A', 'B', 'C', 'D']

for i in incorrect_answers:
    # Make a copy of the incorrect answer list so we don't destroy the original
    i = i.copy()
    
    # Make a temporary correct answer value so we can access it again
    correct_answer = questions['correct_answer'].values[count]
    
    # Append the correct answer to the wrong answer and then shuffle
    i.append(correct_answer)
    random.shuffle(i)
    incorrect.append(i)
    
    # Record the correct answer number for easier use later
    correct_answer_number.append(answer_choices[i.index(correct_answer)])
    count += 1

# Create unique ids for each of the questions
quest_ids = [unique_id(7) for _ in range(questions.shape[0])]

# Append some of the extra stuff from above into th elist
questions[['A', 'B', 'C', 'D']] = pd.DataFrame(incorrect, index=questions.index)
questions.insert(0, 'question_id', quest_ids)
questions.insert(5, 'correct_number', correct_answer_number)
questions

,question_id,category,type,difficulty,question,correct_number,correct_answer,A,B,C,D,incorrect_answers
0,GHV7VPF,Entertainment: Video Games,multiple,medium,"What name did &quot;Mario&quot;, from &quot;Su...",B,Ossan,Mario,Ossan,Jumpman,Mr. Video,"[Jumpman, Mr. Video, Mario]"
1,QV46HNS,Entertainment: Japanese Anime & Manga,multiple,medium,Who wrote and directed the animated movie &quo...,C,Hayao Miyazaki,Mamoru Hosoda,Hidetaka Miyazaki,Hayao Miyazaki,Isao Takahata,"[Isao Takahata, Mamoru Hosoda, Hidetaka Miyazaki]"
2,XNMZYQN,Entertainment: Video Games,multiple,medium,What&#039;s the famous line Vaas says in &quot...,D,Did I ever tell you the definition of Insanity?,You&#039;re my b*tch!,Have I failed to entertain you?,Maybe your best course...would be to tread lig...,Did I ever tell you the definition of Insanity?,"[Have I failed to entertain you?, You&#039;re ..."
3,NYJ4ZPC,Geography,multiple,easy,What state is the largest state of the United ...,B,Alaska,Washington,Alaska,Texas,California,"[California, Texas, Washington]"
4,3J6SM5R,Entertainment: Video Games,boolean,hard,Unturned originally started as a Roblox game.,B,True,False,True,None,None,[False]
...,...,...,...,...,...,...,...,...,...,...,...,...
45,Z4BCDE9,Geography,multiple,medium,Which of these Japanese islands is the largest...,A,Shikoku,Shikoku,Odaiba,Okinawa,Iki,"[Iki, Odaiba, Okinawa]"
46,7D5U5HU,Geography,multiple,hard,"Bir Tawil, an uninhabited track of land claime...",A,Egypt and Sudan,Egypt and Sudan,Iraq and Saudi Arabia,Chad and Libya,Israel and Jordan,"[Israel and Jordan, Chad and Libya, Iraq and S..."
47,NF4W34F,Entertainment: Music,multiple,medium,Which Beatle wrote and sang the song &quot;Why...,B,Paul,Ringo,Paul,John,George,"[John, George, Ringo]"
48,5WMS34D,Entertainment: Video Games,multiple,medium,Who was Tetris created by?,D,Alexey Pajitnov,Allan Alcorn,William Higinbotham,Toru Iwatani,Alexey Pajitnov,"[Toru Iwatani, Allan Alcorn, William Higinbotham]"


Remove duplicates based on columns category, type, difficulty, and question since there may have been duplicates when the questions were randomly generated and downloaded from the API (dropping only based on question may be enough, but we will drop using all these attributes just to be safe and make sure all duplicates are indeed dropped).

In [5]:
questions = questions.drop_duplicates(subset=['category', 'type', 'difficulty', 'question'], keep='first')
questions

,question_id,category,type,difficulty,question,correct_number,correct_answer,A,B,C,D,incorrect_answers
0,GHV7VPF,Entertainment: Video Games,multiple,medium,"What name did &quot;Mario&quot;, from &quot;Su...",B,Ossan,Mario,Ossan,Jumpman,Mr. Video,"[Jumpman, Mr. Video, Mario]"
1,QV46HNS,Entertainment: Japanese Anime & Manga,multiple,medium,Who wrote and directed the animated movie &quo...,C,Hayao Miyazaki,Mamoru Hosoda,Hidetaka Miyazaki,Hayao Miyazaki,Isao Takahata,"[Isao Takahata, Mamoru Hosoda, Hidetaka Miyazaki]"
2,XNMZYQN,Entertainment: Video Games,multiple,medium,What&#039;s the famous line Vaas says in &quot...,D,Did I ever tell you the definition of Insanity?,You&#039;re my b*tch!,Have I failed to entertain you?,Maybe your best course...would be to tread lig...,Did I ever tell you the definition of Insanity?,"[Have I failed to entertain you?, You&#039;re ..."
3,NYJ4ZPC,Geography,multiple,easy,What state is the largest state of the United ...,B,Alaska,Washington,Alaska,Texas,California,"[California, Texas, Washington]"
4,3J6SM5R,Entertainment: Video Games,boolean,hard,Unturned originally started as a Roblox game.,B,True,False,True,None,None,[False]
...,...,...,...,...,...,...,...,...,...,...,...,...
26,VZ3KKRB,Entertainment: Film,multiple,medium,Darth Vader&#039;s famous reveal to Luke is ic...,C,&quot;No. I am your father.&quot;,&quot;The truth is that I am your father.&quot;,&quot;You&#039;re wrong. I am your father.&quot;,&quot;No. I am your father.&quot;,"&quot;Luke, I am your father.&quot;","[&quot;Luke, I am your father.&quot;, &quot;Yo..."
30,AEQADBY,History,multiple,easy,To what political party did Abraham Lincoln be...,C,Republican,Independent,Whig,Republican,Democrat,"[Democrat, Independent, Whig]"
31,U8U99SU,Entertainment: Video Games,multiple,medium,"What is the name of the common, gun-toting ene...",C,Sligs,Scrabs,Slogs,Sligs,Glukkons,"[Scrabs, Slogs, Glukkons]"
36,KHJVJ82,Entertainment: Japanese Anime & Manga,multiple,easy,"In &quot;To Love-Ru&quot;, Ren and Run are fro...",B,Memorze,Deviluke,Memorze,Mistletoe,Okiwana,"[Deviluke, Mistletoe, Okiwana]"


## Save the datafame as a csv to the data path

In [6]:
questions.to_csv(f'{path}/trivia_questions_combined.csv')

The HTML special characters were cleaned up in a separate editor. The first few rows of the cleaned up version are shown below.

In [ ]:
questions = pd.read_csv(f'{path}/trivia_questions_combined.csv')
questions = questions.drop(columns=['correct_answer', 'incorrect_answers'])
questions = questions.rename(columns={"correct_number":"correct_answer", "type":"question_type"})

In [22]:
questions.head()

,question_id,category,question_type,difficulty,question,correct_answer,A,B,C,D
0,SHRY2XC,Entertainment: Video Games,multiple,medium,"What name did ""Mario"", from ""Super Mario Broth...",C,Mr. Video,Mario,Ossan,Jumpman
1,F4W7KV9,Entertainment: Japanese Anime & Manga,multiple,medium,"Who wrote and directed the animated movie ""Spi...",B,Mamoru Hosoda,Hayao Miyazaki,Isao Takahata,Hidetaka Miyazaki
2,4CWEVSZ,Entertainment: Video Games,multiple,medium,"What's the famous line Vaas says in ""Far Cry 3""?",B,You're my b*tch!,Did I ever tell you the definition of Insanity?,Have I failed to entertain you?,Maybe your best course...would be to tread lig...
3,TMVMBAG,Geography,multiple,easy,What state is the largest state of the United ...,B,California,Alaska,Washington,Texas
4,6GHTTFB,Entertainment: Video Games,boolean,hard,Unturned originally started as a Roblox game.,A,True,False,NaN,NaN


In [23]:
questions.to_csv(f'{path}/trivia_questions_combined.csv')